# exp001_baseline inference

Notebook-first inference for the deterministic last-known TVT baseline.

## Contents

1. Setup and configuration
2. Submission generation


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd

from baseline import HORIZONTAL_SUFFIX, predict_from_prefix, primary_strategy, well_id_from_path
from settings import EXPERIMENT_NAME, ExperimentPaths, load_config

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()
primary = primary_strategy(config)

print("Experiment:", EXPERIMENT_NAME)
print("Root:", paths.root)
print("Test data:", paths.test_data_dir)
print("Sample submission:", paths.sample_submission_path)
print("Submission path:", paths.submission_path)
print("Primary strategy:", primary)
print("Debug:", DEBUG)

## 2. Submission generation


In [ ]:
def test_files(paths: ExperimentPaths) -> list[Path]:
    files = sorted(paths.test_data_dir.glob(f"*{HORIZONTAL_SUFFIX}"))
    if not files:
        raise FileNotFoundError(f"no test horizontal well CSVs found in {paths.test_data_dir}")
    return files


if not paths.sample_submission_path.exists():
    raise FileNotFoundError(f"sample submission not found: {paths.sample_submission_path}")

predictions: dict[str, float] = {}
well_summaries: list[dict[str, object]] = []
for path in test_files(paths):
    well_id = well_id_from_path(path)
    df = pd.read_csv(path)
    prediction = predict_from_prefix(df, config)
    y_pred = prediction.predictions[primary]
    for row_index, value in zip(prediction.eval_indices, y_pred, strict=True):
        predictions[f"{well_id}_{int(row_index)}"] = float(value)
    well_summaries.append(
        {
            "well_id": well_id,
            "n_rows": int(len(df)),
            "n_eval": int(prediction.eval_indices.size),
            "last_known_index": prediction.last_known_index,
            "last_known_tvt": prediction.last_known_tvt,
            "recent_slope": prediction.recent_slope,
        }
    )

sample_submission = pd.read_csv(paths.sample_submission_path)
id_column = config["data"]["id_column"]
target_column = config["data"]["submission_target_column"]
missing_ids = sorted(set(sample_submission[id_column]) - set(predictions))
if missing_ids:
    preview = ", ".join(missing_ids[:5])
    raise ValueError(f"missing predictions for {len(missing_ids)} sample ids: {preview}")

output = sample_submission.copy()
output[target_column] = output[id_column].map(predictions).astype(float)
submission_path = paths.submission_path
output.to_csv(submission_path, index=False)
print("Created Kaggle submission:", submission_path)
print("Predicted rows:", len(predictions))